# 04 — Negotiation & Coordination
**Agentic X | Studio 67**

### Overview
This notebook tests a **multi-round negotiation coordinator** that collects independent decisions from **Creative, Finance, Safety and Operations**. A proposal is accepted only when all four approve and its execution deadline remains feasible. Objections prompt an alternative proposal; unresolved cases are escalated to a human manager.

### Negotiation & Communication
The coordinator uses ACL-like messages (`CFP`, `PROPOSE`, `REQUEST`, `INFORM`, `ACCEPT`, `REFUSE`) with message IDs, sender/receiver, round, and the `studio67.event_crisis` ontology. The system tests **Screen Failure**, **Budget Overrun** and **Safety Conflict**, including conflicts, deadlines, and failures.

### Related Python Files
- `core/negotiation.py` — multi-round coordination, voting, deadlines, and escalation.
- `core/communication.py` — validated ACL message bus.
- `core/models.py` — `AgentEvaluation`, `ACLMessage`, `RunResult`.
- `core/environment.py` — validated 150-case/450-option dataset.
- `agents/*.py` — four independent agent evaluations.
- `core/runner.py` — integrated entry point.

### Requirements & Setup
Python 3.10+ and repository `requirements.txt`. **Real Creative and Operations evaluations require a Gemini key** in Google Colab Secrets named `GOOGLE_API_KEY` (or secure masked input). No key is committed to GitHub. Run cells in order after uploading the updated `core/negotiation.py` to GitHub.

**Testing note:** Controlled tests are explicitly identified and do **not** represent Gemini evaluations. Live tests below call the real agents and may produce different valid outcomes between runs. Screenshots of these outputs can support the project report.

### Cell 1 — Project Setup & Latest Repository

In [1]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path('/content/AgenticX')
REPO_URL = 'https://github.com/ProjectsSpaceHub/AgenticX.git'
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
else:
    if not (ROOT / '.git').exists():
        raise RuntimeError('Project directory exists but is not a Git checkout')
    changes = subprocess.run(['git','-C',str(ROOT),'status','--porcelain'], capture_output=True, text=True, check=True).stdout.strip()
    if changes:
        print('Local changes detected: skipping git pull. Source-version check below will verify required code.')
    else:
        subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'], check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements.txt')],check=True)
print('Repository ready:',ROOT)

Repository ready: /content/AgenticX


### Cell 2 — Import Modules & Verify Updated Sources

In [2]:
import importlib
import pandas as pd
from IPython.display import display
from core.environment import load_dataset, get_case
from core.models import AgentEvaluation, ACLMessage, RunResult
from core.communication import MessageBus, VALID_PERFORMATIVES, ONTOLOGY
import core.negotiation as negotiation_module
importlib.reload(negotiation_module)
NegotiationEngine=negotiation_module.NegotiationEngine
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent
from agents.creative import CreativeAgent
from agents.operations import OperationsAgent
from core.runner import build_engine, run_case
assert NegotiationEngine.VERSION == '2026-11-negotiation-v2', 'Update core/negotiation.py on GitHub first'
assert ONTOLOGY=='studio67.event_crisis'
assert NegotiationEngine.REQUIRED_AGENTS=={'Creative','Finance','Safety','Operations'}
print('Modules verified:',negotiation_module.__file__)

Modules verified: /content/AgenticX/core/negotiation.py


### Cell 3 — Dataset & Scenario Selection

In [3]:
cases_df,options_df=load_dataset()
assert len(cases_df)==150 and len(options_df)==450
scenario_ids=['SCR-001','BDG-001','SFT-001']
scenario_rows=[]
for case_id in scenario_ids:
    c,opts=get_case(case_id)
    assert len(opts)==3 and len({o['option_id'] for o in opts})==3
    scenario_rows.append({'case_id':case_id,'scenario':c['scenario'],'incident':c['incident'],'time_left_min':c['time_left_min'],'options':len(opts)})
scenario_df=pd.DataFrame(scenario_rows)
display(scenario_df)
print('150 cases / 450 options validated')

,case_id,scenario,incident,time_left_min,options
0,SCR-001,Screen Failure,Main LED screen failed during final rehearsal,100,3
1,BDG-001,Budget Overrun,Supplier costs may exceed the event spending plan,80,3
2,SFT-001,Safety Conflict,Emergency exit partially blocked by stage setup,60,3


150 cases / 450 options validated


### Cell 4 — Negotiation Variables & Decision Rules

In [4]:
parameters_df=pd.DataFrame([
 ('max_rounds','Coordinator','Maximum distinct proposals (default 3)','Stops attempts and triggers escalation'),
 ('time_left_min','Case','Minutes remaining until event deadline','Decreases as negotiation takes time'),
 ('duration_min','Option','Minutes needed to execute proposal','Must fit within remaining time'),
 ('quality_score','Option','Recorded solution quality','Influences initial proposal order; Creative also evaluates quality'),
 ('cost_sar','Option','Implementation cost in SAR','Finance judgment; affects alternative ranking'),
 ('gross_savings_sar','Option','Expected gross savings in SAR','Affects financial ranking after objections'),
 ('risk_score_after_mitigation','Option','Residual safety risk','Affects safety ranking; Safety enforces veto'),
 ('approved','AgentEvaluation','Individual independent approval','All four agents must approve'),
 ('blocking','AgentEvaluation','Whether agent issues a hard-veto signal','Recorded for explanation; unanimity still required'),
 ('ontology','ACLMessage','Shared meaning of messages','Ensures consistent message interpretation'),
],columns=['Parameter','Source','Purpose','Effect on behavior'])
display(parameters_df)
print('Decision: unanimous approval + deadline; otherwise alternative or escalation')

,Parameter,Source,Purpose,Effect on behavior
0,max_rounds,Coordinator,Maximum distinct proposals (default 3),Stops attempts and triggers escalation
1,time_left_min,Case,Minutes remaining until event deadline,Decreases as negotiation takes time
2,duration_min,Option,Minutes needed to execute proposal,Must fit within remaining time
3,quality_score,Option,Recorded solution quality,Influences initial proposal order; Creative al...
4,cost_sar,Option,Implementation cost in SAR,Finance judgment; affects alternative ranking
5,gross_savings_sar,Option,Expected gross savings in SAR,Affects financial ranking after objections
6,risk_score_after_mitigation,Option,Residual safety risk,Affects safety ranking; Safety enforces veto
7,approved,AgentEvaluation,Individual independent approval,All four agents must approve
8,blocking,AgentEvaluation,Whether agent issues a hard-veto signal,Recorded for explanation; unanimity still requ...
9,ontology,ACLMessage,Shared meaning of messages,Ensures consistent message interpretation


Decision: unanimous approval + deadline; otherwise alternative or escalation


### Cell 5 — Controlled Test Agents — Isolated Deterministic Tests

In [5]:
# These are EXPLICIT TEST DOUBLES for repeatable coordinator tests, NOT real Gemini agents.
class ControlledAgent:
    def __init__(self,name,declined_options=(),error=False):
        self.name=name
        self.declined_options=set(declined_options)
        self.error=error
        self.calls=[]
    def evaluate(self,case,option):
        self.calls.append(option['option_id'])
        if self.error: raise RuntimeError('controlled agent failure')
        approved=option['option_id'] not in self.declined_options
        return AgentEvaluation(agent=self.name,option_id=option['option_id'],approved=approved,
          score=100 if approved else 0,blocking=self.name in {'Finance','Safety','Operations'},
          reason='Controlled test decision: '+('accepted' if approved else 'refused'))

def controlled_engine(declines=None,fail_role=None,max_rounds=3):
    declines=declines or {}
    agents={name:ControlledAgent(name,declines.get(name,()),error=(name==fail_role))
            for name in ['Creative','Finance','Safety','Operations']}
    engine=NegotiationEngine(agents['Creative'],agents['Finance'],agents['Safety'],agents['Operations'],max_rounds=max_rounds)
    return engine,agents

control_case={'case_id':'CONTROL-01','time_left_min':120}
control_options=[
 {'option_id':'C1','quality_score':95,'duration_min':10,'cost_sar':100,'gross_savings_sar':0,'safety_ok':True,'permit_ok':True,'mitigation_verified':True,'risk_score_after_mitigation':1},
 {'option_id':'C2','quality_score':85,'duration_min':8,'cost_sar':90,'gross_savings_sar':100,'safety_ok':True,'permit_ok':True,'mitigation_verified':True,'risk_score_after_mitigation':1},
 {'option_id':'C3','quality_score':70,'duration_min':6,'cost_sar':80,'gross_savings_sar':0,'safety_ok':True,'permit_ok':True,'mitigation_verified':True,'risk_score_after_mitigation':1},
]
print('Controlled test fixtures ready (no Gemini calls)')

Controlled test fixtures ready (no Gemini calls)


### Cell 6 — Direct Agreement & Unanimous Approval

In [6]:
control_engine,agents=controlled_engine()
agreement=control_engine.run(control_case,control_options)
agreement.is_mock=True  # Label the controlled result truthfully
assert agreement.status=='AGREED' and agreement.rounds==1
assert agreement.selected_option_id=='C1'
assert len(agreement.evaluations)==4
assert {e.agent for e in agreement.evaluations}==NegotiationEngine.REQUIRED_AGENTS
assert all(e.approved for e in agreement.evaluations)
assert any(m.performative=='ACCEPT' and m.sender=='Coordinator' for m in agreement.messages)
print('Controlled direct agreement passed:',agreement.status,agreement.selected_option_id)

Controlled direct agreement passed: AGREED C1


### Cell 7 — Finance Objection & Alternative Proposal

In [7]:
control_engine,agents=controlled_engine({'Finance':{'C1'}})
conflict=control_engine.run(control_case,control_options)
conflict.is_mock=True
assert conflict.status=='AGREED' and conflict.rounds==2
assert conflict.selected_option_id=='C2'
assert any(e.agent=='Finance' and not e.approved and e.option_id=='C1' for e in conflict.evaluations)
assert {e.option_id for e in conflict.evaluations}=={'C1','C2'}
assert len({m.content for m in conflict.messages if m.performative=='PROPOSE'})==2
print('Controlled objection -> round 2 alternative passed')
display(pd.DataFrame([{'round':m.round,'sender':m.sender,'performative':m.performative,'content':m.content} for m in conflict.messages if m.performative in ('CFP','PROPOSE','REFUSE','ACCEPT')]).head(18))

Controlled objection -> round 2 alternative passed


,round,sender,performative,content
0,1,Coordinator,CFP,"Call for proposals, round 1"
1,1,Coordinator,PROPOSE,Proposed option: C1
2,1,Creative,ACCEPT,C1: Controlled test decision: accepted
3,1,Finance,REFUSE,C1: Controlled test decision: refused
4,1,Safety,ACCEPT,C1: Controlled test decision: accepted
5,1,Operations,ACCEPT,C1: Controlled test decision: accepted
6,2,Coordinator,CFP,"Call for proposals, round 2"
7,2,Coordinator,PROPOSE,Proposed option: C2
8,2,Creative,ACCEPT,C2: Controlled test decision: accepted
9,2,Finance,ACCEPT,C2: Controlled test decision: accepted


### Cell 8 — Safety Veto & Human Escalation

In [8]:
control_engine,agents=controlled_engine({'Safety':{'C1','C2','C3'}})
escalation=control_engine.run(control_case,control_options)
escalation.is_mock=True
assert escalation.status=='ESCALATED' and escalation.rounds==3
assert escalation.selected_option_id is None
assert len({e.option_id for e in escalation.evaluations})==3
assert any(m.receiver=='Human Manager' and m.performative=='REQUEST' for m in escalation.messages)
assert all(not e.approved for e in escalation.evaluations if e.agent=='Safety')
print('Controlled safety veto and escalation passed')

Controlled safety veto and escalation passed


### Cell 9 — Deadline Changes & No-Agreement Handling

In [9]:
assert NegotiationEngine._deadline_check(10,12,30)[0]
assert not NegotiationEngine._deadline_check(10,10,5)[0]
assert not NegotiationEngine._deadline_check(-1,10,0)[0]
assert not NegotiationEngine._deadline_check(4,0,0)[0]
short_case={**control_case,'time_left_min':0}
control_engine,_=controlled_engine()
time_out=control_engine.run(short_case,control_options)
time_out.is_mock=True
assert time_out.status=='ESCALATED'
assert time_out.rounds==3
assert any('Deadline failed' in m.content for m in time_out.messages)
assert any(m.receiver=='Human Manager' for m in time_out.messages)
print('Deadline and zero-time escalation passed')

Deadline and zero-time escalation passed


### Cell 10 — Agent Failure, Duplicate & Empty Option Tests

In [10]:
control_engine,_=controlled_engine(fail_role='Finance')
failure=control_engine.run(control_case,control_options)
failure.is_mock=True
assert failure.status=='FAILED' and failure.selected_option_id is None
assert any(m.receiver=='Human Manager' for m in failure.messages)
control_engine,_=controlled_engine()
try:
    control_engine.run(control_case,[control_options[0],control_options[0]])
except ValueError as exc:
    assert 'Duplicate' in str(exc)
else: raise AssertionError('Duplicate proposals were not rejected')
empty=control_engine.run(control_case,[])
empty.is_mock=True
assert empty.status=='ESCALATED' and empty.rounds==0
assert any(m.receiver=='Human Manager' for m in empty.messages)
print('Agent failure, duplicate and empty proposals passed')

Agent failure, duplicate and empty proposals passed


### Cell 11 — ACL Message Integrity & Protocol Coverage

In [11]:
required_types={'CFP','PROPOSE','REQUEST','INFORM','ACCEPT','REFUSE'}
recorded_types={m.performative for result in (agreement,conflict,escalation,time_out,failure) for m in result.messages}
assert required_types.issubset(recorded_types)
for result in (agreement,conflict,escalation,time_out,failure):
    ids=[m.message_id for m in result.messages]
    assert len(ids)==len(set(ids))
    assert all(m.case_id==result.case_id and m.ontology==ONTOLOGY for m in result.messages)
    index={m.message_id:m for m in result.messages}
    for m in result.messages:
        if m.in_reply_to:
            original=index[m.in_reply_to]
            assert m.sender==original.receiver and m.receiver==original.sender
    assert [m.round for m in result.messages]==sorted(m.round for m in result.messages)
assert all(m.sender=='Coordinator' for m in agreement.messages if m.performative=='CFP')
print('ACL protocol, ontology, sender/receiver and reply links validated')
print('Observed:',', '.join(sorted(recorded_types)))

ACL protocol, ontology, sender/receiver and reply links validated
Observed: ACCEPT, CFP, INFORM, PROPOSE, REFUSE, REQUEST


### Cell 12 — Negotiation Behavior Under Each Dataset Scenario

In [12]:
# No fabricated agent evaluations: this cell only inspects three REAL dataset cases
# and their deterministic priority ordering (not Gemini judgments).
priority_rows=[]
priority_engine,_=controlled_engine()
for case_id in scenario_ids:
    c,opts=get_case(case_id)
    ranked=sorted(opts,key=lambda o:priority_engine._priority(o,set()),reverse=True)
    priority_rows.extend({'case_id':case_id,'scenario':c['scenario'],'rank':i,
      'option_id':o['option_id'],'quality_score':o['quality_score'],
      'duration_min':o['duration_min'],'cost_sar':o['cost_sar']}
      for i,o in enumerate(ranked,1))
priority_df=pd.DataFrame(priority_rows)
display(priority_df)
assert len(priority_df)==9
print('Three real scenarios inspected; this is proposal ranking, not agent approval')

,case_id,scenario,rank,option_id,quality_score,duration_min,cost_sar
0,SCR-001,Screen Failure,1,SCR-001-O1,89,40,1290
1,SCR-001,Screen Failure,2,SCR-001-O2,89,70,5920
2,SCR-001,Screen Failure,3,SCR-001-O3,60,25,1210
3,BDG-001,Budget Overrun,1,BDG-001-O3,99,20,6890
4,BDG-001,Budget Overrun,2,BDG-001-O2,82,50,2480
5,BDG-001,Budget Overrun,3,BDG-001-O1,64,30,1150
6,SFT-001,Safety Conflict,1,SFT-001-O3,85,65,3140
7,SFT-001,Safety Conflict,2,SFT-001-O2,75,30,1810
8,SFT-001,Safety Conflict,3,SFT-001-O1,67,10,350


Three real scenarios inspected; this is proposal ranking, not agent approval


### Cell 14 — Gemini API Key & Live Agents

**Note — Secure Gemini API Configuration:** Add your own `GOOGLE_API_KEY` in Google Colab Secrets and enable notebook access, or enter it securely when prompted. The key is intentionally excluded from GitHub. The following cells use real Gemini evaluations; API availability and model responses can vary.

In [14]:
import getpass
try:
    from google.colab import userdata
    secret=userdata.get('GOOGLE_API_KEY')
except (ImportError,KeyError,RuntimeError):
    secret=None
if not secret:
    secret=getpass.getpass('Gemini API key (hidden): ')
if not secret or not secret.strip():
    raise RuntimeError('A Gemini API key is required for real negotiations')
os.environ['GOOGLE_API_KEY']=secret.strip()
del secret
os.environ.setdefault('AGENTICX_MODEL','gemini-3.5-flash-lite')
creative_agent=CreativeAgent()
finance_agent=FinanceAgent()
safety_agent=SafetyAgent()
operations_agent=OperationsAgent()
live_engine=NegotiationEngine(creative_agent,finance_agent,safety_agent,operations_agent,max_rounds=3)
print('Four REAL agents initialized; model:',os.environ['AGENTICX_MODEL'])
print('API key is not displayed or saved')

Four REAL agents initialized; model: gemini-3.5-flash-lite
API key is not displayed or saved


### Cell 14 — Real Multi-Agent Negotiation — Three Scenarios

In [15]:
# LIVE evaluation: LLM calls are made to Creative and (when gates pass) Operations.
# Outcome is OBSERVED; do not force agreement or claim a fixed number of rounds.
import time
live_results={}
live_rows=[]
for case_id in scenario_ids:
    case,solutions=get_case(case_id)
    started=time.perf_counter()
    result=live_engine.run(case,solutions)
    duration_ms=round((time.perf_counter()-started)*1000,2)
    assert isinstance(result,RunResult)
    assert result.case_id==case_id and result.status in {'AGREED','ESCALATED','FAILED'}
    assert 0<=result.rounds<=3 and result.is_mock is False
    if result.status=='AGREED':
        approved=[e for e in result.evaluations if e.option_id==result.selected_option_id]
        assert len(approved)==4 and all(e.approved for e in approved)
        assert any(m.sender=='Coordinator' and m.performative=='ACCEPT' for m in result.messages)
    if result.status in {'FAILED','ESCALATED'}:
        assert result.selected_option_id is None
        assert any(m.receiver=='Human Manager' for m in result.messages)
    live_results[case_id]=result
    live_rows.append({'case_id':case_id,'scenario':case['scenario'],'status':result.status,
      'selected_option_id':result.selected_option_id,'rounds':result.rounds,
      'evaluations':len(result.evaluations),'messages':len(result.messages),
      'duration_ms':duration_ms,'reason':result.reason})
live_summary_df=pd.DataFrame(live_rows)
display(live_summary_df)
print('Three LIVE scenario runs completed; inspect observed outcomes')

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

,case_id,scenario,status,selected_option_id,rounds,evaluations,messages,duration_ms,reason
0,SCR-001,Screen Failure,AGREED,SCR-001-O1,1,4,15,15517.32,All four agents approved; execution deadline p...
1,BDG-001,Budget Overrun,ESCALATED,None,3,12,46,53016.66,No unanimous feasible proposal within the nego...
2,SFT-001,Safety Conflict,AGREED,SFT-001-O2,3,12,46,42053.47,All four agents approved; execution deadline p...


Three LIVE scenario runs completed; inspect observed outcomes


### Cell 15 — Real Agent Decisions & ACL Trace

In [16]:
real_evaluation_rows=[]
real_message_rows=[]
for case_id,result in live_results.items():
    for ev in result.evaluations:
        real_evaluation_rows.append({'case_id':case_id,**ev.model_dump()})
    for msg in result.messages:
        real_message_rows.append(msg.model_dump())
real_evaluations_df=pd.DataFrame(real_evaluation_rows,columns=['case_id','agent','option_id','approved','score','reason','blocking'])
real_messages_df=pd.DataFrame(real_message_rows,columns=list(ACLMessage.model_fields))
for case_id,result in live_results.items():
    assert len({m.message_id for m in result.messages})==len(result.messages)
    assert all(m.ontology==ONTOLOGY for m in result.messages)
    assert len(result.evaluations)%4==0
print('Real agent assessments:',len(real_evaluations_df),'| ACL messages:',len(real_messages_df))
display(real_evaluations_df[['case_id','agent','option_id','approved','score','blocking']])
display(real_messages_df[['case_id','round','sender','receiver','performative','in_reply_to']].head(24))

Real agent assessments: 28 | ACL messages: 107


,case_id,agent,option_id,approved,score,blocking
0,SCR-001,Creative,SCR-001-O1,True,89.0,False
1,SCR-001,Finance,SCR-001-O1,True,100.0,True
2,SCR-001,Safety,SCR-001-O1,True,100.0,True
3,SCR-001,Operations,SCR-001-O1,True,100.0,True
4,BDG-001,Creative,BDG-001-O3,False,0.0,False
5,BDG-001,Finance,BDG-001-O3,False,0.0,True
6,BDG-001,Safety,BDG-001-O3,True,100.0,True
7,BDG-001,Operations,BDG-001-O3,True,100.0,True
8,BDG-001,Creative,BDG-001-O2,False,30.0,False
9,BDG-001,Finance,BDG-001-O2,True,100.0,True


,case_id,round,sender,receiver,performative,in_reply_to
0,SCR-001,1,Coordinator,ALL,CFP,None
1,SCR-001,1,Coordinator,ALL,PROPOSE,None
2,SCR-001,1,Coordinator,Creative,REQUEST,None
3,SCR-001,1,Creative,Coordinator,INFORM,SCR-001-M0003
4,SCR-001,1,Creative,ALL,ACCEPT,None
5,SCR-001,1,Coordinator,Finance,REQUEST,None
6,SCR-001,1,Finance,Coordinator,INFORM,SCR-001-M0006
7,SCR-001,1,Finance,ALL,ACCEPT,None
8,SCR-001,1,Coordinator,Safety,REQUEST,None
9,SCR-001,1,Safety,Coordinator,INFORM,SCR-001-M0009


### Cell 16 — Integrated Runner & Source Consistency

In [17]:
# Importing the original runner should still work with the updated coordinator.
assert build_engine.__module__=='core.runner'
assert NegotiationEngine.VERSION=='2026-11-negotiation-v2'
assert all(callable(getattr(x,'evaluate',None)) for x in live_engine.agents)
assert Path(negotiation_module.__file__).resolve()==(ROOT/'core'/'negotiation.py').resolve()
assert 'def run(' in (ROOT/'core'/'negotiation.py').read_text(encoding='utf-8')
print('Core negotiation module is imported from GitHub clone; no source rewriting')

Core negotiation module is imported from GitHub clone; no source rewriting


### Cell 17 — Export Real Results for Project Report

In [18]:
results_dir=ROOT/'results'
results_dir.mkdir(exist_ok=True)
live_summary_df.to_csv(results_dir/'negotiation_live_summary.csv',index=False)
real_evaluations_df.to_csv(results_dir/'negotiation_live_evaluations.csv',index=False)
real_messages_df.to_csv(results_dir/'negotiation_acl_messages.csv',index=False)
priority_df.to_csv(results_dir/'negotiation_scenario_priority.csv',index=False)
print('Saved real negotiation outputs in:',results_dir)
print('Only LIVE data was saved as negotiation_live_*; controlled test results remain separate')

Saved real negotiation outputs in: /content/AgenticX/results
Only LIVE data was saved as negotiation_live_*; controlled test results remain separate


### Cell 19 — Conclusion

The coordinator combines four independent agent decisions in an adaptive, maximum-three-round negotiation. It sends structured ACL messages, considers objections when choosing alternative proposals, checks time remaining, and requests human review when no unanimous feasible proposal exists or an agent fails.

Three realistic event-crisis scenarios are executed using the **real agents**. Controlled tests independently verify agreement, financial objection, safety veto, deadline failure, and escalation. The final UI, end-to-end evaluation, presentation evidence, and report remain in Notebook 05 and the project deliverables.